In [1]:
import os
import json
import time
import queue
import sqlite3
import random
import socket
import threading
import hashlib
import secrets
import tkinter as tk
from tkinter import messagebox, filedialog

# ===================== ML (External) =====================
import numpy as np
from sklearn.linear_model import LogisticRegression
import pickle

# ===================== Windows SAPI (Voice) =====================
try:
    import pythoncom
    import win32com.client
    HAS_SAPI = True
except Exception:
    HAS_SAPI = False

# ===================== OPTIONAL: Pillow for JPG background =====================
PIL_OK = False
try:
    from PIL import Image, ImageTk
    PIL_OK = True
except Exception:
    PIL_OK = False

# ===================== CONFIG =====================
APP_TITLE = "Neon Rock Paper Scissors (ML AI)"
DB_PATH = "users.db"

BG_IMAGE_PATH = "istockphoto-1560833158-612x612.jpg"  # change if needed

# ✅ Bigger window so everything fits nicely
WINDOW_W = 640
WINDOW_H = 760
MIN_W = 580
MIN_H = 700

# Neon theme
BG_SOLID = "#070A14"
NEON_BLUE = "#00E5FF"
NEON_PINK = "#FF2CDF"
NEON_PURPLE = "#7C3AED"
NEON_GREEN = "#22C55E"
NEON_ORANGE = "#F97316"
TEXT = "#E5E7EB"
MUTED = "#94A3B8"

CHOICES = ["rock", "paper", "scissors"]

# ML training thresholds
ML_MIN_ROUNDS = 60
ML_RETRAIN_EVERY = 15
MODEL_DIR = "models"


# ===================== SAPI WORKER (FIXED) =====================
if HAS_SAPI:
    class SAPIWorker:
        """
        Single worker thread for SAPI voice.
        Prevents COM speak errors by initializing COM inside the worker thread.
        """
        def __init__(self, rate=0, volume=100):
            self.q = queue.Queue()
            self.rate = rate
            self.volume = volume
            self.running = True
            self.thread = threading.Thread(target=self._loop, daemon=True)
            self.thread.start()

        def _loop(self):
            pythoncom.CoInitialize()
            try:
                speaker = win32com.client.Dispatch("SAPI.SpVoice")
                speaker.Rate = self.rate
                speaker.Volume = self.volume

                while self.running:
                    try:
                        text = self.q.get(timeout=0.25)
                    except queue.Empty:
                        continue
                    if text is None:
                        continue
                    try:
                        speaker.Speak(str(text))
                    except Exception as e:
                        print("SAPI speak error:", e)
            finally:
                pythoncom.CoUninitialize()

        def speak(self, text: str):
            if text:
                self.q.put(text)

        def stop(self):
            self.running = False
            try:
                self.q.put(None)
            except Exception:
                pass
else:
    class SAPIWorker:
        def __init__(self, rate=0, volume=100):
            self.rate = rate
            self.volume = volume

        def speak(self, text: str):
            pass

        def stop(self):
            pass


# ===================== SOUND (Beep) =====================
def play_sound(kind="click"):
    try:
        import winsound
        if kind == "win":
            winsound.Beep(880, 120); winsound.Beep(988, 120)
        elif kind == "lose":
            winsound.Beep(220, 180); winsound.Beep(196, 200)
        elif kind == "draw":
            winsound.Beep(440, 120)
        else:
            winsound.Beep(600, 60)
    except Exception:
        pass


# ===================== GAME LOGIC =====================
def determine_winner(p1, p2):
    if p1 == p2:
        return "draw"
    rules = {"rock": "scissors", "paper": "rock", "scissors": "paper"}
    return "p1" if rules[p1] == p2 else "p2"


# ===================== DATABASE (SQLite) =====================
def db_init():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            salt TEXT NOT NULL,
            pw_hash TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS user_moves (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL,
            p1_move TEXT NOT NULL,
            p2_move TEXT NOT NULL,
            result TEXT NOT NULL,
            ts TEXT NOT NULL
        )
    """)

    con.commit()
    con.close()


def hash_password(password, salt_hex):
    salt = bytes.fromhex(salt_hex)
    dk = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, 150_000)
    return dk.hex()


def signup_user(username, password):
    username = username.strip()
    if not username or not password:
        return False, "Username এবং Password দুটোই লাগবে।"
    if len(password) < 4:
        return False, "Password কমপক্ষে 4 character দিন।"

    salt_hex = secrets.token_hex(16)
    pw_hash = hash_password(password, salt_hex)

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    try:
        cur.execute(
            "INSERT INTO users (username, salt, pw_hash, created_at) VALUES (?,?,?,?)",
            (username, salt_hex, pw_hash, time.strftime("%Y-%m-%d %H:%M:%S"))
        )
        con.commit()
        return True, "Signup সফল ✅ এখন Login করুন।"
    except sqlite3.IntegrityError:
        return False, "এই Username আগে থেকেই আছে। অন্যটা দিন।"
    finally:
        con.close()


def login_user(username, password):
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("SELECT salt, pw_hash FROM users WHERE username=?", (username.strip(),))
    row = cur.fetchone()
    con.close()
    if not row:
        return False, "User পাওয়া যায়নি। আগে Signup করুন।"
    salt_hex, stored = row
    if hash_password(password, salt_hex) == stored:
        return True, "Login সফল ✅"
    return False, "Password ভুল।"


# ===================== NEON BUTTON =====================
class NeonButton(tk.Button):
    def __init__(self, master, base_bg, hover_bg, *args, **kwargs):
        super().__init__(master, *args, **kwargs)
        self.base_bg = base_bg
        self.hover_bg = hover_bg
        self.configure(
            bg=self.base_bg,
            fg="white",
            activebackground=self.hover_bg,
            activeforeground="white",
            bd=0,
            relief="flat",
            cursor="hand2",
            font=("Segoe UI", 11, "bold"),
            padx=14, pady=8
        )
        self._pulse_on = False
        self._pulse_state = 0
        self.bind("<Enter>", self._on_enter)
        self.bind("<Leave>", self._on_leave)

    def start_pulse(self):
        self._pulse_on = True
        self._pulse_tick()

    def stop_pulse(self):
        self._pulse_on = False
        self.configure(bg=self.base_bg)

    def _pulse_tick(self):
        if not self._pulse_on:
            return
        self._pulse_state = 1 - self._pulse_state
        self.configure(bg=self.hover_bg if self._pulse_state else self.base_bg)
        self.after(650, self._pulse_tick)

    def _on_enter(self, _):
        self.configure(bg=self.hover_bg)

    def _on_leave(self, _):
        self.configure(bg=self.base_bg)


# ===================== ONLINE MULTIPLAYER =====================
class NetSession:
    def __init__(self):
        self.sock = None
        self.is_host = False
        self.peer_user = "Player2"
        self.inbox = queue.Queue()
        self.running = False
        self.thread = None

    def _recv_loop(self):
        buf = b""
        try:
            while self.running:
                data = self.sock.recv(4096)
                if not data:
                    break
                buf += data
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    if line.strip():
                        try:
                            msg = json.loads(line.decode("utf-8"))
                            self.inbox.put(msg)
                        except Exception:
                            pass
        finally:
            self.running = False
            try:
                self.sock.close()
            except Exception:
                pass
            self.sock = None
            self.inbox.put({"type": "disconnected"})

    def send(self, msg: dict):
        try:
            if self.sock:
                self.sock.sendall((json.dumps(msg) + "\n").encode("utf-8"))
        except Exception:
            pass

    def host(self, port: int, username: str):
        self.is_host = True
        srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        srv.bind(("", port))
        srv.listen(1)

        self.inbox.put({"type": "status", "text": f"Hosting... (port {port}) Waiting for player..."})
        client, _addr = srv.accept()
        srv.close()

        self.sock = client
        self.running = True
        self.thread = threading.Thread(target=self._recv_loop, daemon=True)
        self.thread.start()
        self.send({"type": "hello", "user": username})

    def join(self, host_ip: str, port: int, username: str):
        self.is_host = False
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.connect((host_ip, port))
        self.sock = s
        self.running = True
        self.thread = threading.Thread(target=self._recv_loop, daemon=True)
        self.thread.start()
        self.send({"type": "hello", "user": username})

    def close(self):
        self.running = False
        try:
            if self.sock:
                self.send({"type": "quit"})
                self.sock.close()
        except Exception:
            pass
        self.sock = None


# ===================== ML HELPERS =====================
MOVE_TO_IDX = {"rock": 0, "paper": 1, "scissors": 2}
IDX_TO_MOVE = {0: "rock", 1: "paper", 2: "scissors"}
RES_TO_IDX = {"win": 0, "lose": 1, "draw": 2}

def one_hot3(idx):
    v = np.zeros(3, dtype=np.float32)
    if 0 <= idx < 3:
        v[idx] = 1.0
    return v

def model_path_for_user(username: str) -> str:
    os.makedirs(MODEL_DIR, exist_ok=True)
    safe = "".join([c for c in username if c.isalnum() or c in ("_", "-")])[:50] or "user"
    return os.path.join(MODEL_DIR, f"{safe}_rps_model.pkl")


# ===================== APP =====================
class App:
    def __init__(self, root):
        self.root = root
        self.root.title(APP_TITLE)
        self.root.geometry(f"{WINDOW_W}x{WINDOW_H}")
        self.root.minsize(MIN_W, MIN_H)
        self.root.resizable(True, True)

        db_init()

        # ✅ Fixed voice system (single worker)
        self.voice = SAPIWorker(rate=0, volume=100)
        self.speak("Text to speech is ready.")

        self.username = ""
        self.net = None

        self.mode = None
        self.reset_scores()

        # background state
        self.bg_label = None
        self.bg_photo = None
        self.bg_raw = None
        self.overlay = None
        self._bg_last_wh = (0, 0)

        self.ml_model = None
        self.ml_ready = False
        self.ml_training = False
        self.ml_last_trained_count = 0
        self.ml_lock = threading.Lock()
        self.ml_state_text = "AI: warmup (need 60+ rounds to train)"

        # ✅ resize-safe background
        self.root.bind("<Configure>", self._on_resize)

        self.show_login()

    def speak(self, text: str):
        try:
            self.voice.speak(text)
        except Exception:
            pass

    def reset_scores(self):
        self.p1_score = 0
        self.p2_score = 0
        self.draws = 0
        self.pending_online_p1 = None
        self.pending_online_p2 = None
        self.hand_anim_token = 0

    # ---------- UI Helpers ----------
    def clear(self):
        for w in self.root.winfo_children():
            w.destroy()
        # reset background widgets too
        self.bg_label = None
        self.bg_photo = None
        self.overlay = None
        self._bg_last_wh = (0, 0)

    def _load_bg_raw(self):
        global BG_IMAGE_PATH
        if BG_IMAGE_PATH and os.path.exists(BG_IMAGE_PATH) and PIL_OK:
            try:
                self.bg_raw = Image.open(BG_IMAGE_PATH).convert("RGB")
            except Exception:
                self.bg_raw = None
        else:
            self.bg_raw = None

    def _draw_bg(self, w: int, h: int):
        # create/refresh background image scaled to current window size
        if not PIL_OK or self.bg_raw is None or w <= 1 or h <= 1:
            if self.bg_label:
                self.bg_label.destroy()
                self.bg_label = None
            self.root.configure(bg=BG_SOLID)
            return

        try:
            img = self.bg_raw.resize((w, h))
            self.bg_photo = ImageTk.PhotoImage(img)

            if not self.bg_label:
                self.bg_label = tk.Label(self.root, image=self.bg_photo, bd=0)
                self.bg_label.place(x=0, y=0, relwidth=1, relheight=1)
            else:
                self.bg_label.configure(image=self.bg_photo)

            if not self.overlay:
                self.overlay = tk.Frame(self.root, bg=BG_SOLID)
                self.overlay.place(x=0, y=0, relwidth=1, relheight=1)
                self.overlay.configure(highlightthickness=0)
            else:
                self.overlay.lift()
        except Exception:
            self.root.configure(bg=BG_SOLID)

    def _on_resize(self, event):
        # avoid too many redraws
        w = int(self.root.winfo_width())
        h = int(self.root.winfo_height())
        if abs(w - self._bg_last_wh[0]) < 10 and abs(h - self._bg_last_wh[1]) < 10:
            return
        self._bg_last_wh = (w, h)

        if self.bg_raw is not None and self.bg_label is not None:
            # redraw scaled bg
            self._draw_bg(w, h)

    def set_background(self):
        self.root.configure(bg=BG_SOLID)
        self._load_bg_raw()
        w = int(self.root.winfo_width() or WINDOW_W)
        h = int(self.root.winfo_height() or WINDOW_H)
        self._draw_bg(w, h)

    def card(self):
        # ✅ Bigger card so elements never clip
        frame = tk.Frame(self.root, bg="#0B1022", highlightthickness=1, highlightbackground="#182044")
        frame.place(relx=0.5, rely=0.5, anchor="center", width=520, height=620)
        return frame

    def header(self, parent, text, color=NEON_PINK, size=24):
        tk.Label(parent, text=text, fg=color, bg=parent["bg"], font=("Segoe UI", size, "bold")).pack(pady=(18, 8))

    def small(self, parent, text, color=MUTED):
        tk.Label(parent, text=text, fg=color, bg=parent["bg"], font=("Segoe UI", 10)).pack()

    # ---------- LOGIN / SIGNUP ----------
    def show_login(self):
        self.clear()
        self.set_background()
        frame = self.card()

        self.header(frame, "LOGIN", NEON_BLUE, 26)
        self.small(frame, "Please enter your username and password to log in")

        tk.Label(frame, text="Username", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(pady=(18, 4))
        self.login_user = tk.Entry(frame, font=("Segoe UI", 12))
        self.login_user.pack(ipady=6, padx=50, fill="x")

        tk.Label(frame, text="Password", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(pady=(12, 4))
        self.login_pass = tk.Entry(frame, font=("Segoe UI", 12), show="*")
        self.login_pass.pack(ipady=6, padx=50, fill="x")

        btn_login = NeonButton(frame, NEON_PINK, NEON_PURPLE, text="Login", command=self.do_login)
        btn_login.pack(pady=(18, 10))
        btn_login.start_pulse()

        btn_signup = NeonButton(frame, NEON_GREEN, NEON_BLUE, text="Go to Signup", command=self.show_signup)
        btn_signup.pack()

        btn_bg = NeonButton(frame, NEON_ORANGE, NEON_PINK, text="Change Background", command=self.pick_bg)
        btn_bg.pack(pady=(14, 0))

    def show_signup(self):
        self.clear()
        self.set_background()
        frame = self.card()

        self.header(frame, "SIGNUP", NEON_PINK, 26)
        self.small(frame, "নতুন account বানান (SQLite database এ save হবে)")

        tk.Label(frame, text="Username", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(pady=(18, 4))
        self.sup_user = tk.Entry(frame, font=("Segoe UI", 12))
        self.sup_user.pack(ipady=6, padx=50, fill="x")

        tk.Label(frame, text="Password", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(pady=(12, 4))
        self.sup_pass = tk.Entry(frame, font=("Segoe UI", 12), show="*")
        self.sup_pass.pack(ipady=6, padx=50, fill="x")

        btn_create = NeonButton(frame, NEON_PURPLE, NEON_PINK, text="Create Account", command=self.do_signup)
        btn_create.pack(pady=(18, 10))
        btn_create.start_pulse()

        btn_back = NeonButton(frame, NEON_BLUE, NEON_GREEN, text="Back to Login", command=self.show_login)
        btn_back.pack()

    def do_signup(self):
        play_sound("click")
        u = self.sup_user.get().strip()
        p = self.sup_pass.get()
        ok, msg = signup_user(u, p)
        if ok:
            play_sound("win")
            self.speak("Signup successful. Please login.")
            messagebox.showinfo("Signup", msg)
            self.show_login()
        else:
            play_sound("lose")
            self.speak("Signup failed.")
            messagebox.showerror("Signup Failed", msg)

    def do_login(self):
        play_sound("click")
        u = self.login_user.get().strip()
        p = self.login_pass.get()
        ok, msg = login_user(u, p)
        if ok:
            play_sound("win")
            self.username = u
            self.speak(f"Login successful. Welcome {u}.")
            self.load_model_for_user()
            self.show_menu()
        else:
            play_sound("lose")
            self.speak("Login failed.")
            messagebox.showerror("Login Failed", msg)

    def pick_bg(self):
        path = filedialog.askopenfilename(
            title="Pick Background Image",
            filetypes=[("Images", "*.png *.gif *.jpg *.jpeg *.webp"), ("All", "*.*")]
        )
        if path:
            global BG_IMAGE_PATH
            BG_IMAGE_PATH = path
            self.show_login()

    # ---------- MAIN MENU ----------
    def show_menu(self):
        self.clear()
        self.set_background()
        frame = self.card()

        self.header(frame, "GAME MENU", NEON_PINK, 24)
        tk.Label(frame, text=f"Welcome, {self.username}",
                 fg=NEON_BLUE, bg=frame["bg"], font=("Segoe UI", 12, "bold")).pack(pady=(0, 8))
        self.small(frame, "Please select Mode")

        NeonButton(frame, NEON_PURPLE, NEON_PINK, text="Player vs Computer (ML AI)",
                   command=lambda: self.start_game("PVC")).pack(pady=(18, 10))

        NeonButton(frame, NEON_GREEN, NEON_BLUE, text="Local 2 Players (Same PC)",
                   command=lambda: self.start_game("LOCAL2")).pack(pady=(0, 10))

        NeonButton(frame, NEON_ORANGE, NEON_PINK, text="Online Multiplayer (LAN)",
                   command=self.show_online_lobby).pack(pady=(0, 10))

        NeonButton(frame, NEON_BLUE, NEON_GREEN, text="Logout",
                   command=self.logout).pack(pady=(14, 0))

    def logout(self):
        if self.net:
            self.net.close()
            self.net = None
        self.username = ""
        self.reset_scores()
        self.ml_model = None
        self.ml_ready = False
        self.ml_training = False
        self.ml_last_trained_count = 0
        self.ml_state_text = "AI: warmup (need 60+ rounds to train)"
        self.speak("Logged out.")
        self.show_login()

    # ---------- ONLINE LOBBY ----------
    def show_online_lobby(self):
        self.clear()
        self.set_background()
        frame = self.card()

        self.header(frame, "ONLINE MULTIPLAYER", NEON_BLUE, 20)
        self.small(frame, "Same Wi-Fi/LAN এ Host/Join করুন (Socket)")

        tk.Label(frame, text="Host", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 12, "bold")).pack(pady=(18, 6))
        host_row = tk.Frame(frame, bg=frame["bg"])
        host_row.pack(padx=40, fill="x")
        tk.Label(host_row, text="Port", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(side="left")
        self.host_port = tk.Entry(host_row, font=("Segoe UI", 11))
        self.host_port.insert(0, "5050")
        self.host_port.pack(side="left", padx=10, fill="x", expand=True)

        NeonButton(frame, NEON_PURPLE, NEON_PINK, text="Start Hosting",
                   command=self.online_host).pack(pady=(10, 16))

        tk.Label(frame, text="Join", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 12, "bold")).pack(pady=(0, 6))
        join_row = tk.Frame(frame, bg=frame["bg"])
        join_row.pack(padx=40, fill="x")
        tk.Label(join_row, text="IP", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(side="left")
        self.join_ip = tk.Entry(join_row, font=("Segoe UI", 11))
        self.join_ip.insert(0, "127.0.0.1")
        self.join_ip.pack(side="left", padx=10, fill="x", expand=True)

        join_row2 = tk.Frame(frame, bg=frame["bg"])
        join_row2.pack(padx=40, pady=8, fill="x")
        tk.Label(join_row2, text="Port", fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11)).pack(side="left")
        self.join_port = tk.Entry(join_row2, font=("Segoe UI", 11))
        self.join_port.insert(0, "5050")
        self.join_port.pack(side="left", padx=10, fill="x", expand=True)

        NeonButton(frame, NEON_GREEN, NEON_BLUE, text="Join Game",
                   command=self.online_join).pack(pady=(6, 18))

        self.net_status = tk.Label(frame, text="", fg=MUTED, bg=frame["bg"], font=("Segoe UI", 10))
        self.net_status.pack()

        NeonButton(frame, NEON_ORANGE, NEON_PINK, text="Back",
                   command=self.show_menu).pack(pady=(18, 0))

    def online_host(self):
        play_sound("click")
        port = int(self.host_port.get().strip() or "5050")
        self.net = NetSession()
        self.speak("Hosting game.")

        def run_host():
            try:
                self.net.host(port, self.username)
            except Exception as e:
                self.net.inbox.put({"type": "error", "text": str(e)})

        threading.Thread(target=run_host, daemon=True).start()
        self.root.after(100, self.poll_net_lobby)

    def online_join(self):
        play_sound("click")
        ip = self.join_ip.get().strip()
        port = int(self.join_port.get().strip() or "5050")
        self.net = NetSession()
        self.speak("Joining game.")

        def run_join():
            try:
                self.net.join(ip, port, self.username)
            except Exception as e:
                self.net.inbox.put({"type": "error", "text": str(e)})

        threading.Thread(target=run_join, daemon=True).start()
        self.root.after(100, self.poll_net_lobby)

    def poll_net_lobby(self):
        if not self.net:
            return
        try:
            while True:
                msg = self.net.inbox.get_nowait()
                t = msg.get("type")
                if t == "status":
                    self.net_status.config(text=msg.get("text", ""))
                elif t == "hello":
                    self.net.peer_user = msg.get("user", "Player2")
                    self.speak(f"Connected with {self.net.peer_user}.")
                    self.start_game("ONLINE")
                    return
                elif t == "error":
                    messagebox.showerror("Network Error", msg.get("text", "Error"))
                    self.speak("Network error.")
                    self.net = None
                    return
                elif t == "disconnected":
                    self.net_status.config(text="Disconnected.")
        except queue.Empty:
            pass
        self.root.after(100, self.poll_net_lobby)

    # ---------- GAME SCREEN ----------
    def start_game(self, mode):
        self.mode = mode
        self.reset_scores()

        self.clear()
        self.set_background()
        frame = self.card()

        title = "Rock Paper Scissors"
        if mode == "PVC":
            subtitle = f"{self.username} vs ML AI"
        elif mode == "LOCAL2":
            subtitle = f"{self.username} vs Player 2 (Same PC)"
        else:
            subtitle = f"{self.username} vs {self.net.peer_user if self.net else 'Online Player'}"

        self.header(frame, title, NEON_PINK, 22)
        tk.Label(frame, text=subtitle, fg=NEON_BLUE, bg=frame["bg"],
                 font=("Segoe UI", 11, "bold")).pack(pady=(0, 10))

        self.status = tk.Label(frame, text="Pick a move!",
                               fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11))
        self.status.pack()

        self.moves_label = tk.Label(frame, text="P1: -   |   P2: -",
                                    fg=NEON_BLUE, bg=frame["bg"], font=("Segoe UI", 12, "bold"))
        self.moves_label.pack(pady=(10, 6))

        self.hand_area = tk.Frame(frame, bg=frame["bg"], width=420, height=140)
        self.hand_area.pack(pady=(6, 4))
        self.hand_area.pack_propagate(False)

        self.hand_left = tk.Label(self.hand_area, text="✊", fg=TEXT, bg=frame["bg"],
                                  font=("Segoe UI Emoji", 46))
        self.hand_right = tk.Label(self.hand_area, text="✊", fg=TEXT, bg=frame["bg"],
                                   font=("Segoe UI Emoji", 46))
        self.hand_left.place(x=120, y=50, anchor="center")
        self.hand_right.place(x=300, y=50, anchor="center")

        self.result_label = tk.Label(frame, text="",
                                     fg=TEXT, bg=frame["bg"], font=("Segoe UI", 14, "bold"))
        self.result_label.pack(pady=(6, 8))

        btn_row = tk.Frame(frame, bg=frame["bg"])
        btn_row.pack(pady=(16, 10))

        self.btn_rock = NeonButton(btn_row, NEON_GREEN, NEON_BLUE, text="Rock",
                                   command=lambda: self.play("rock"))
        self.btn_paper = NeonButton(btn_row, NEON_PURPLE, NEON_PINK, text="Paper",
                                    command=lambda: self.play("paper"))
        self.btn_scis = NeonButton(btn_row, NEON_PINK, NEON_ORANGE, text="Scissors",
                                   command=lambda: self.play("scissors"))

        self.btn_rock.grid(row=0, column=0, padx=10)
        self.btn_paper.grid(row=0, column=1, padx=10)
        self.btn_scis.grid(row=0, column=2, padx=10)

        self.btn_rock.start_pulse()
        self.btn_paper.start_pulse()
        self.btn_scis.start_pulse()

        self.score_label = tk.Label(frame, text=self.score_text(),
                                    fg=TEXT, bg=frame["bg"], font=("Segoe UI", 11, "bold"))
        self.score_label.pack(pady=(14, 8))

        bottom = tk.Frame(frame, bg=frame["bg"])
        bottom.pack(pady=(10, 0))

        NeonButton(bottom, NEON_BLUE, NEON_GREEN, text="Reset", command=self.reset_round).grid(row=0, column=0, padx=10)
        NeonButton(bottom, NEON_ORANGE, NEON_PINK, text="Back", command=self.back_from_game).grid(row=0, column=1, padx=10)
        NeonButton(bottom, "#EF4444", NEON_PINK, text="Quit", command=self.quit_app).grid(row=0, column=2, padx=10)

        self.ml_line = tk.Label(frame, text="", fg=MUTED, bg=frame["bg"], font=("Segoe UI", 9, "italic"))
        self.ml_line.pack(pady=(10, 0))
        self.refresh_ml_line()

        tip = "PVC: আপনার move pattern শিখে AI counter move দিবে।"
        if mode == "LOCAL2":
            tip = "LOCAL2: (voice + UI works) Player2 choose from popup."
        if mode == "ONLINE":
            tip = "ONLINE: your move sent to opponent."
            self.root.after(100, self.poll_net_game)

        tk.Label(frame, text=tip, fg=MUTED, bg=frame["bg"], font=("Segoe UI", 10, "italic")).pack(pady=(8, 0))

        if mode == "PVC":
            self.speak("Player versus computer.")
        elif mode == "LOCAL2":
            self.speak("Local two player mode.")
        else:
            self.speak("Online multiplayer mode.")

    def refresh_ml_line(self):
        if self.mode == "PVC":
            self.ml_line.config(text=self.ml_state_text)
        self.root.after(600, self.refresh_ml_line)

    def back_from_game(self):
        if self.net and self.mode == "ONLINE":
            try:
                self.net.send({"type": "quit"})
            except Exception:
                pass
            self.net.close()
            self.net = None
        self.show_menu()

    def quit_app(self):
        if messagebox.askyesno("Quit", "Do you want to quit?"):
            try:
                if self.net:
                    self.net.close()
            finally:
                try:
                    self.voice.stop()
                except Exception:
                    pass
                self.root.destroy()

    def reset_round(self):
        play_sound("click")
        self.reset_scores()
        self.status.config(text="Pick a move!")
        self.moves_label.config(text="P1: -   |   P2: -")
        self.result_label.config(text="")
        self.score_label.config(text=self.score_text())
        if self.mode == "ONLINE" and self.net:
            self.net.send({"type": "reset"})
        self.speak("Round reset.")

    def score_text(self):
        return f"Score | P1: {self.p1_score}  P2: {self.p2_score}  Draws: {self.draws}"

    # ---------- PLAY ----------
    def play(self, p1_move):
        play_sound("click")

        if self.mode == "PVC":
            try:
                p2_move = self.ai_choose_move_from_history()
            except Exception as e:
                print("AI choose error:", e)
                p2_move = random.choice(CHOICES)

            self.resolve_round(p1_move, p2_move, p2_name="ML AI")

            try:
                self.ai_record_and_maybe_train(p1_move, p2_move)
            except Exception as e:
                print("AI record/train error:", e)
            return

        if self.mode == "LOCAL2":
            p2_move = self.ask_p2_move()
            if not p2_move:
                return
            self.resolve_round(p1_move, p2_move, p2_name="Player2")
            return

        if self.mode == "ONLINE":
            if not self.net:
                messagebox.showerror("Online", "Not connected.")
                return
            self.pending_online_p1 = p1_move
            self.status.config(text="Waiting for opponent...")
            self.net.send({"type": "move", "move": p1_move})

    def ask_p2_move(self):
        top = tk.Toplevel(self.root)
        top.title("Player2 Move")
        top.geometry("320x240")
        top.configure(bg=BG_SOLID)
        top.resizable(False, False)
        top.grab_set()

        choice_var = tk.StringVar(value="")

        tk.Label(top, text="Player 2: Choose your move",
                 fg=NEON_BLUE, bg=BG_SOLID, font=("Segoe UI", 11, "bold")).pack(pady=18)

        def pick(v):
            choice_var.set(v)
            top.destroy()

        row = tk.Frame(top, bg=BG_SOLID)
        row.pack(pady=12)

        NeonButton(row, NEON_GREEN, NEON_BLUE, text="Rock", command=lambda: pick("rock")).grid(row=0, column=0, padx=8)
        NeonButton(row, NEON_PURPLE, NEON_PINK, text="Paper", command=lambda: pick("paper")).grid(row=0, column=1, padx=8)
        NeonButton(row, NEON_PINK, NEON_ORANGE, text="Scissors", command=lambda: pick("scissors")).grid(row=0, column=2, padx=8)

        top.wait_window()

        mv = choice_var.get() or None
        if mv:
            self.speak(f"Player two chose {mv}.")
        return mv

    def resolve_round(self, p1_move, p2_move, p2_name="Player2"):
        self.moves_label.config(text=f"P1: {p1_move}   |   P2: {p2_move}")
        self.animate_hands(p1_move, p2_move)

        self.speak(f"You chose {p1_move}. Opponent chose {p2_move}.")

        w = determine_winner(p1_move, p2_move)
        if w == "p1":
            self.p1_score += 1
            self.result_label.config(text="✅ Player 1 Wins!", fg=NEON_GREEN)
            self.status.config(text="Nice! Keep going!")
            play_sound("win")
            self.speak("You win.")
            res = "win"
        elif w == "p2":
            self.p2_score += 1
            self.result_label.config(text=f"❌ {p2_name} Wins!", fg=NEON_PINK)
            self.status.config(text="Oops! Try again.")
            play_sound("lose")
            self.speak("You lose.")
            res = "lose"
        else:
            self.draws += 1
            self.result_label.config(text="➖ Draw!", fg=TEXT)
            self.status.config(text="Same move! Again?")
            play_sound("draw")
            self.speak("It is a draw.")
            res = "draw"

        self.score_label.config(text=self.score_text())
        self.last_player_result = res

    def animate_hands(self, p1_move, p2_move):
        if not hasattr(self, "hand_area"):
            return
        self.hand_anim_token += 1
        token = self.hand_anim_token

        def move_to_emoji(mv):
            return {"rock": "✊", "paper": "🖐", "scissors": "✌"}.get(mv, "✊")

        self.hand_left.config(text="✊")
        self.hand_right.config(text="✊")

        self.hand_area.update_idletasks()
        area_w = int(self.hand_area.winfo_width() or 420)
        left_x = int(area_w * 0.30)
        right_x = int(area_w * 0.70)
        base_y = 55

        offsets = [0, -12, 12, -10, 10, -8, 8, -6, 6, -3, 3, 0]
        delay = 70

        def tick(i):
            if token != self.hand_anim_token:
                return
            dx = offsets[i]
            self.hand_left.place(x=left_x + dx, y=base_y, anchor="center")
            self.hand_right.place(x=right_x - dx, y=base_y, anchor="center")
            if i + 1 < len(offsets):
                self.root.after(delay, lambda: tick(i + 1))
            else:
                self.hand_left.config(text=move_to_emoji(p1_move))
                self.hand_right.config(text=move_to_emoji(p2_move))

        tick(0)

    def poll_net_game(self):
        if not self.net or self.mode != "ONLINE":
            return
        try:
            while True:
                msg = self.net.inbox.get_nowait()
                t = msg.get("type")

                if t == "hello":
                    self.net.peer_user = msg.get("user", "Player2")
                elif t == "move":
                    self.pending_online_p2 = msg.get("move")
                    if self.pending_online_p1 and self.pending_online_p2:
                        p1 = self.pending_online_p1
                        p2 = self.pending_online_p2
                        self.pending_online_p1 = None
                        self.pending_online_p2 = None
                        self.resolve_round(p1, p2, p2_name=self.net.peer_user)
                        self.status.config(text="Pick a move!")
                elif t == "reset":
                    self.reset_round()
                elif t == "quit" or t == "disconnected":
                    messagebox.showinfo("Online", "Opponent disconnected.")
                    self.speak("Opponent disconnected.")
                    self.net.close()
                    self.net = None
                    self.show_menu()
                    return
        except queue.Empty:
            pass

        self.root.after(100, self.poll_net_game)

    # ===================== ML CORE =====================
    def load_model_for_user(self):
        self.ml_model = None
        self.ml_ready = False
        self.ml_training = False
        self.ml_last_trained_count = 0

        path = model_path_for_user(self.username)
        if os.path.exists(path):
            try:
                with open(path, "rb") as f:
                    obj = pickle.load(f)
                if isinstance(obj, dict) and "model" in obj:
                    self.ml_model = obj["model"]
                    self.ml_last_trained_count = int(obj.get("trained_count", 0))
                    self.ml_ready = True
                    self.ml_state_text = f"AI: model loaded ✅ (trained_count={self.ml_last_trained_count})"
                else:
                    self.ml_state_text = "AI: warmup (need 60+ rounds to train)"
            except Exception as e:
                print("Model load error:", e)
                self.ml_state_text = "AI: warmup (need 60+ rounds to train)"
        else:
            self.ml_state_text = "AI: warmup (need 60+ rounds to train)"

    def get_user_moves_from_db(self, limit=None):
        con = sqlite3.connect(DB_PATH)
        cur = con.cursor()
        if limit is None:
            cur.execute("SELECT p1_move, result FROM user_moves WHERE username=? ORDER BY id ASC", (self.username,))
        else:
            cur.execute("SELECT p1_move, result FROM user_moves WHERE username=? ORDER BY id DESC LIMIT ?",
                        (self.username, int(limit)))
        rows = cur.fetchall()
        con.close()
        if limit is not None:
            rows.reverse()
        return rows

    def get_user_move_count(self) -> int:
        con = sqlite3.connect(DB_PATH)
        cur = con.cursor()
        cur.execute("SELECT COUNT(*) FROM user_moves WHERE username=?", (self.username,))
        n = int(cur.fetchone()[0])
        con.close()
        return n

    def build_feature_vector_from_history(self, history):
        if len(history) < 3:
            return None

        moves = [m for (m, _r) in history]
        results = [_r for (_m, _r) in history]

        last3 = moves[-3:]
        last_res = results[-1] if results else "draw"

        feat = []
        for mv in last3:
            feat.append(one_hot3(MOVE_TO_IDX.get(mv, -1)))
        feat = np.concatenate(feat, axis=0)

        rvec = one_hot3(RES_TO_IDX.get(last_res, 2))

        streak = 1
        for i in range(len(moves)-2, -1, -1):
            if moves[i] == moves[-1]:
                streak += 1
            else:
                break
        streak = min(streak, 10)
        streak_vec = np.array([streak / 10.0], dtype=np.float32)

        last10 = moves[-10:] if len(moves) >= 10 else moves[:]
        counts = [last10.count("rock"), last10.count("paper"), last10.count("scissors")]
        total = max(1, len(last10))
        dist = np.array([c/total for c in counts], dtype=np.float32)

        x = np.concatenate([feat, rvec, streak_vec, dist], axis=0).reshape(1, -1)
        return x

    def make_dataset(self):
        rows = self.get_user_moves_from_db(limit=None)
        if len(rows) < 4:
            return None, None, len(rows)

        X_list = []
        y_list = []

        for t in range(3, len(rows)):
            hist = rows[:t]
            x = self.build_feature_vector_from_history(hist)
            if x is None:
                continue
            y_move = rows[t][0]
            if y_move not in MOVE_TO_IDX:
                continue
            X_list.append(x[0])
            y_list.append(MOVE_TO_IDX[y_move])

        if not X_list:
            return None, None, len(rows)

        X = np.array(X_list, dtype=np.float32)
        y = np.array(y_list, dtype=np.int64)
        return X, y, len(rows)

    def train_model_async(self):
        if self.ml_training:
            return

        def worker():
            try:
                with self.ml_lock:
                    self.ml_training = True
                self.ml_state_text = "AI: training... ⏳"

                X, y, total_rows = self.make_dataset()
                if X is None or y is None or len(y) < 10:
                    self.ml_state_text = f"AI: warmup (need more usable data) | rounds={total_rows}"
                    return

                model = LogisticRegression(max_iter=400, solver="lbfgs")
                model.fit(X, y)

                path = model_path_for_user(self.username)
                with open(path, "wb") as f:
                    pickle.dump({"model": model, "trained_count": total_rows}, f)

                with self.ml_lock:
                    self.ml_model = model
                    self.ml_ready = True
                    self.ml_last_trained_count = total_rows

                self.ml_state_text = f"AI: trained ✅ (rounds={total_rows})"
                self.speak("Artificial intelligence trained.")
            except Exception as e:
                print("Training error:", e)
                self.ml_state_text = f"AI: training failed ({e})"
            finally:
                with self.ml_lock:
                    self.ml_training = False

        threading.Thread(target=worker, daemon=True).start()

    def ai_choose_move_from_history(self):
        history = self.get_user_moves_from_db(limit=None)
        rounds = len(history)

        if rounds < ML_MIN_ROUNDS:
            self.ml_state_text = f"AI: warmup (need {ML_MIN_ROUNDS}+ rounds) | current={rounds}"
            return random.choice(CHOICES)

        if not self.ml_ready and not self.ml_training:
            self.train_model_async()
            return random.choice(CHOICES)

        with self.ml_lock:
            model = self.ml_model
            ready = self.ml_ready

        if not ready or model is None:
            return random.choice(CHOICES)

        x = self.build_feature_vector_from_history(history)
        if x is None:
            return random.choice(CHOICES)

        pred_idx = int(model.predict(x)[0])
        pred_move = IDX_TO_MOVE.get(pred_idx, random.choice(CHOICES))

        counter = {"rock": "paper", "paper": "scissors", "scissors": "rock"}
        return counter.get(pred_move, random.choice(CHOICES))

    def ai_record_and_maybe_train(self, p1_move, p2_move):
        w = determine_winner(p1_move, p2_move)
        res = "win" if w == "p1" else ("lose" if w == "p2" else "draw")

        con = sqlite3.connect(DB_PATH)
        cur = con.cursor()
        cur.execute(
            "INSERT INTO user_moves (username, p1_move, p2_move, result, ts) VALUES (?,?,?,?,?)",
            (self.username, p1_move, p2_move, res, time.strftime("%Y-%m-%d %H:%M:%S"))
        )
        con.commit()
        con.close()

        total = self.get_user_move_count()
        if total >= ML_MIN_ROUNDS:
            if (not self.ml_ready) and (not self.ml_training):
                self.train_model_async()
            elif self.ml_ready and (total - self.ml_last_trained_count) >= ML_RETRAIN_EVERY and (not self.ml_training):
                self.train_model_async()


# ===================== RUN =====================
if __name__ == "__main__":
    root = tk.Tk()
    app = App(root)
    root.mainloop()
